# Part 1 - Campustown EDA & Visualizations

> Case study part: Part 1 of 3
> Required for: All students (3-credit and 4-credit)
> Role: You are a Real Estate Analyst at Quad Capital Partners (QCP).

This notebook is the first deliverable of the campustown case study.
Before we can model prices or recommend acquisitions, the Investment Committee
needs a clear, well-illustrated picture of the sixteen markets in QCP's
footprint. By the end of this notebook you should be able to answer:

1. What is the data, and is it trustworthy?
2. How do the sixteen campus towns compare on price, size, and inventory?
3. Which structural features of a home appear to drive price?
4. Are there any obvious outliers or data-quality issues we need to flag for
   the modeling step in Part 2?

We use Plotly Express for every chart so the deck team can drop the same
figures straight into the slide deliverable.


## ⚙️ Setup

We pin to a small, focused stack: `pandas` and `numpy` for tabular work and
`plotly.express` for visualization.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.io as pio

# House style for every chart in this case study.
pio.templates.default = "simple_white"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)


## 📥 Load the Data

The properties file lives next to this notebook in `data/`. We keep paths
relative so the notebook is portable across machines.


In [ ]:
DATA_DIR = "data"

properties = pd.read_parquet(f"{DATA_DIR}/campustown_properties.parquet")
print(f"Loaded {len(properties):,} property listings")
properties.head(3)


Quick schema check before we touch anything.

In [ ]:
properties.info()


## 🩺 Data-Quality Triage

Before we chart anything, we want a one-page summary of how clean each column
is. For an investment-grade analysis we care about three things:

1. Missingness - which columns have so many nulls they cannot be used as
   features.
2. Implausible values - `$0` listings, `0` bedrooms, future build years.
3. Coverage - does each campus town have enough listings to model on its
   own, or do small markets need to be pooled?


Per-column missingness and dtypes, sorted by % missing.

In [ ]:
quality = (
    pd.DataFrame({
        "n_missing": properties.isnull().sum(),
        "pct_missing": properties.isnull().mean().mul(100).round(1),
        "dtype": properties.dtypes.astype(str),
    })
    .sort_values("pct_missing", ascending=False)
)
quality


:::{tip} What to take away

Anything missing in more than ~30% of rows is risky to use directly as a
feature. The `reso_*` columns (HOA fees, school ratings, architectural style)
are the most affected. We will either drop them or convert them to "present /
absent" flags in Part 2.
:::


A short list of implausible-value sanity checks.

In [ ]:
checks = {
    "price <= 0": (properties["price"] <= 0).sum(),
    "beds == 0": (properties["beds"] == 0).sum(),
    "baths == 0": (properties["baths"] == 0).sum(),
    "living_area missing or 0": (
        properties["living_area"].isnull() | (properties["living_area"] == 0)
    ).sum(),
    "year_built > 2026": (properties["year_built"] > 2026).sum(),
    "year_built < 1800": (properties["year_built"] < 1800).sum(),
}
pd.Series(checks, name="rows_flagged").to_frame()


:::{important} Why we skip `days_on_zillow`

The `days_on_zillow` field is just the gap between the listing's posting date
and the date the data was scraped. It does not tell us how long a listing was
live before it sold, so it is not a reliable liquidity signal for active vs.
stale inventory. We drop it from the EDA. Part 3 derives a much cleaner
days-to-sale measure from the price-history event log instead.
:::


## 🧹 A Light Cleaning Pass

We make a clean copy of the dataset for the rest of the EDA. The rules are
intentionally conservative - we are not trying to impute everything yet,
just to remove rows that would distort summary statistics.


Drop listings that obviously cannot be modeled.

In [ ]:
df = properties.copy()

mask = (
    (df["price"] > 10_000)            # exclude $0 / $1 placeholder listings
    & (df["price"] < 5_000_000)       # exclude trophy outliers (rare in campus towns)
    & (df["beds"].between(1, 10))
    & (df["baths"].between(0.5, 10))
    & (df["living_area"].between(300, 15_000))
)
df = df.loc[mask].copy()
print(f"Kept {len(df):,} of {len(properties):,} listings ({len(df) / len(properties):.1%})")


Convert dates and add a few derived helpers we will reuse throughout.

In [ ]:
df["date_posted"] = pd.to_datetime(df["date_posted"], errors="coerce")
df["price_per_sqft"] = df["price"] / df["living_area"]
df["age_years"] = 2026 - df["year_built"]
df["market"] = df["address_city"] + ", " + df["address_state"]


### 🎓 Map each city to its anchoring university

Every market in QCP's footprint is anchored by a major public university. We
map each `(city, state)` pair to the campus name so we can group satellite
towns (e.g., Middleton with Madison, Carrboro with Chapel Hill, Savoy and
Urbana with Champaign) under their parent university.


In [ ]:
CITY_TO_CAMPUS = {
    ("Madison",         "WI"): "University of Wisconsin-Madison",
    ("Middleton",       "WI"): "University of Wisconsin-Madison",
    ("Ann Arbor",       "MI"): "University of Michigan",
    ("Athens",          "GA"): "University of Georgia",
    ("State College",   "PA"): "Pennsylvania State University",
    ("Charlottesville", "VA"): "University of Virginia",
    ("Gainesville",     "FL"): "University of Florida",
    ("Chapel Hill",     "NC"): "University of North Carolina at Chapel Hill",
    ("Carrboro",        "NC"): "University of North Carolina at Chapel Hill",
    ("Davis",           "CA"): "University of California, Davis",
    ("Iowa City",       "IA"): "University of Iowa",
    ("Champaign",       "IL"): "University of Illinois Urbana-Champaign",
    ("Savoy",           "IL"): "University of Illinois Urbana-Champaign",
    ("Urbana",          "IL"): "University of Illinois Urbana-Champaign",
    ("Lafayette",       "IN"): "Purdue University",
    ("West Lafayette",  "IN"): "Purdue University",
}

df["campus"] = [
    CITY_TO_CAMPUS.get((c, s)) for c, s in zip(df["address_city"], df["address_state"])
]
print("Unmapped rows:", df["campus"].isna().sum())
df["campus"].value_counts().to_frame("listings")


Quick numeric summary of the cleaned frame.

In [ ]:
df[["price", "price_per_sqft", "beds", "baths", "living_area", "age_years"]].describe().round(1)


## 🏙️ Market Coverage

QCP operates in sixteen campus towns across nine universities. We check
that the dataset has a defensible sample size in each one before we trust
market-level comparisons.


In [ ]:
market_counts = (
    df.groupby("market", as_index=False)
      .agg(listings=("property_id", "count"),
           median_price=("price", "median"),
           median_sqft=("living_area", "median"),
           median_psf=("price_per_sqft", "median"))
      .sort_values("listings", ascending=False)
)
market_counts


Listings by city. We use a horizontal bar chart so all sixteen names are readable.

In [ ]:
fig = px.bar(
    market_counts.sort_values("listings"),
    x="listings", y="market", orientation="h",
    title="Listings per campus town",
    labels={"listings": "Number of listings", "market": ""},
    height=600,
)
fig.show()


:::{note} Why a horizontal bar chart?

Sixteen long city names will not fit on a vertical x-axis without rotating
labels into an unreadable angle. Horizontal bars also make it easy for the
viewer to scan top-to-bottom. We set `height=600` so each bar stays readable.
:::


Aggregating to the campus level pools satellite towns with their parent city,
which is closer to how QCP thinks about each market.


In [ ]:
campus_counts = (
    df.dropna(subset=["campus"])
      .groupby("campus", as_index=False)
      .agg(listings=("property_id", "count"),
           median_price=("price", "median"),
           median_psf=("price_per_sqft", "median"))
      .sort_values("listings", ascending=False)
)
campus_counts


In [ ]:
fig = px.bar(
    campus_counts.sort_values("listings"),
    x="listings", y="campus", orientation="h",
    title="Listings per anchoring university",
    labels={"listings": "Number of listings", "campus": ""},
    height=500,
)
fig.show()


## 💰 Price Distribution by Market

Median price is the right central-tendency metric here - campus-town markets
have heavy right tails (a handful of multi-million-dollar faculty trophy
homes per market).


In [ ]:
order = (
    df.groupby("market")["price"].median().sort_values().index.tolist()
)

fig = px.box(
    df, x="market", y="price", category_orders={"market": order},
    title="Single-family list price by campus town (USD)",
    labels={"market": "", "price": "List price (USD)"},
    height=600,
)
fig.update_yaxes(range=[0, 1_500_000])
fig.update_xaxes(tickangle=-30)
fig.show()


Price-per-square-foot ranks markets in a way that controls for home size.

In [ ]:
fig = px.bar(
    market_counts.sort_values("median_psf"),
    x="median_psf", y="market", orientation="h",
    title="Median price per square foot by campus town",
    labels={"median_psf": "Price per sq ft (USD)", "market": ""},
    color="median_psf", color_continuous_scale="Blues",
    height=600,
)
fig.show()


:::{tip} Talking point for the deck

Price-per-square-foot is the metric most acquisition committees default to
because it normalizes away the obvious effect of size. Use this chart on its
own slide and call out the cheapest and most expensive markets.
:::


## 📐 What Drives Price Within a Market?

Living area is the biggest single driver in almost every residential market.
Faceting by market lets the committee see how steep that relationship is in
each town.


In [ ]:
sample = df.sample(n=min(5000, len(df)), random_state=0)
top_markets = market_counts.nlargest(8, "listings")["market"].tolist()
sample8 = sample[sample["market"].isin(top_markets)]

fig = px.scatter(
    sample8, x="living_area", y="price",
    color="beds", facet_col="market", facet_col_wrap=4,
    opacity=0.6, height=700,
    title="Price vs. living area, by market (top 8 by listing count)",
    labels={"living_area": "Living area (sq ft)", "price": "List price (USD)"},
)
fig.update_yaxes(range=[0, 1_500_000])
fig.show()


Price-per-square-foot by bedroom count, pooled across markets.

In [ ]:
fig = px.box(
    df[df["beds"].between(2, 6)],
    x="beds", y="price_per_sqft",
    title="Price per sq ft by bedroom count (across all markets)",
    labels={"beds": "Bedrooms", "price_per_sqft": "Price per sq ft (USD)"},
)
fig.update_yaxes(range=[0, 600])
fig.show()


Reading the chart: smaller homes (2-3 BR) typically have the highest
price per square foot because the kitchen, bathroom, and HVAC fixed costs
are spread over fewer feet. This is a classic real-estate pattern and a sign
that our cleaning step left realistic data behind.


## 🏚️ Age of the Housing Stock

Older housing stock often signals an established neighborhood near campus -
exactly the kind of inventory QCP wants - but it also signals higher
renovation costs. Worth showing the committee.


In [ ]:
fig = px.histogram(
    df[df["age_years"].between(0, 150)],
    x="age_years", nbins=40, color="address_state",
    title="Age of campus-town housing stock (years since construction)",
    labels={"age_years": "Age (years)"},
    barmode="overlay", opacity=0.55,
)
fig.show()


## 🗺️ Geographic Spread

A scatter on a map shows the Investment Committee how concentrated each
campus-town market actually is - most demand is within a ~3-mile radius of
campus.


In [ ]:
geo = df.dropna(subset=["latitude", "longitude"]).sample(
    n=min(8000, len(df)), random_state=0
)

fig = px.scatter_map(
    geo, lat="latitude", lon="longitude",
    color="price", size="living_area",
    hover_data=["address_city", "address_state", "price", "beds", "baths"],
    color_continuous_scale="Viridis", range_color=[100_000, 900_000],
    zoom=3, height=600,
    title="Single-family listings across the QCP footprint",
    map_style="carto-positron",
)
fig.show()


## 📝 Description Length - A Preview of Part 2b

The `description` column is free-text written by the listing agent. We will
mine it with spaCy in Part 2b, but it is worth a quick look right now: very
short descriptions are often distress sales or tear-downs, and very long ones
often signal premium homes. That is already a useful price signal.


In [ ]:
df["desc_word_count"] = (
    df["description"].fillna("").str.split().map(len)
)

fig = px.scatter(
    df.sample(n=min(5000, len(df)), random_state=0),
    x="desc_word_count", y="price", color="market",
    opacity=0.5,
    title="List price vs. description length",
    labels={"desc_word_count": "Words in listing description",
            "price": "List price (USD)"},
)
fig.update_yaxes(range=[0, 1_500_000])
fig.update_xaxes(range=[0, 600])
fig.show()


## 💾 Save the Cleaned Dataset for Part 2

Following the same pattern used in the *Data Job Postings* case, we persist a
cleaned Parquet file so Part 2 can load it directly without repeating any of
the cleaning logic. The `campus` column is included so downstream notebooks
can use it as a feature.


In [ ]:
out_path = f"{DATA_DIR}/properties_clean.parquet"
df.to_parquet(out_path, index=False)
print(f"Saved cleaned dataset -> {out_path}  ({len(df):,} rows)")


## 🧠 Investment-Committee Takeaways (Part 1)

Before moving on to modeling, write up three to five bullets in your final
deliverable that summarize what you would tell the Investment Committee
today, based only on EDA. Examples of the kind of bullet they want:

- "Davis, Ann Arbor, and Charlottesville are the three most expensive markets
  on a $/sqft basis; Lafayette IN and Champaign IL are the cheapest."
- "The University of Illinois Urbana-Champaign campus market is fragmented
  across Champaign, Urbana, and Savoy - pooling them gives us the
  third-largest sample in the footprint."
- "About X% of listings have HOA information missing; we will need a vendor
  source for that field before modeling rental cash flow."

In Part 2a we move from describing the market to predicting price.
